# Full Codebase
For the paper: *Mapping Cerebello-Cortical Structural Connectivity Associated with Cognitive Impairment in Essential Tremor*

By Sajan Patel

sajancpatel@gmail.com

$\color{blue}{\text{AI Disclaimer:}}$

I used Generative AI (Google's Gemini) to help me understand how to set up much of this brain-processing pipeline and create animations.

## Plotting SUIT Cerebellar Atlas
1. Download `'atl-Anatom_space-MNI_dseg.nii'` from https://github.com/DiedrichsenLab/cerebellar_atlases/tree/master/Diedrichsen_2009 and place in current directory.

Imports

In [ ]:
import nibabel as nib
import numpy as np
from nilearn import plotting
import matplotlib.pyplot as plt

Load the atlas

In [ ]:
# Load the atlas
atlas_file = 'atl-Anatom_space-MNI_dseg.nii'
atlas_img = nib.load(atlas_file)

# Extract the raw 3D array data
atlas_data = atlas_img.get_fdata()

Print voxel count for each region in the atlas

In [ ]:
# Find unique region IDs in the data array and count how many voxels belong to each
unique_regions, voxel_counts = np.unique(atlas_data, return_counts=True)

print("Cerebellar Atlas Regions:")
print("-" * 30)
for region_id, count in zip(unique_regions, voxel_counts):
    # 0 is the background
    if region_id == 0:
        print(f"Background (0): {count} voxels")
    else:
        print(f"Region {int(region_id)}: {count} voxels")

Plot the entire atlas in MNI space

In [ ]:
# nilearn's plot_roi automatically overlays the atlas on the MNI152 template
display = plotting.plot_roi(
    atlas_img, 
    title="Whole Cerebellar Atlas (MNI Space)",
    draw_cross=False,
    cmap='Paired' # Good colormap for discrete regions
)
plt.show()

Plot a few regions individually

In [ ]:
regions_to_plot = [3, 4, 14, 16]
for region_id in regions_to_plot: 
    print(f"Plotting Region {int(region_id)}...")
    
    # Create a binary mask that is 1 where the data equals the region_id, and 0 elsewhere
    single_region_data = (atlas_data == region_id).astype(np.int32)
    
    # Create a new nifti image for this single region using the original affine matrix
    single_region_img = nib.Nifti1Image(single_region_data, atlas_img.affine)
    
    # Plot this specific region
    plotting.plot_roi(
        single_region_img, 
        title=f"Region {int(region_id)}",
        draw_cross=True, # Helps center the view on the region
        cmap='autumn'    # Color for the individual region
    )

    plt.show()

## Plotting Peak Voxel Coordinates

Imports

In [ ]:
import pandas as pd
from nilearn import plotting

Load in excel file containing all Peak Voxel Coordinates.

Each tab has a different network.

LOI = Locations of Interest = Peak Voxel Coordinates

In [ ]:
# The single Excel file containing all the tabs
excel_file = "loi_coordinates.xlsx"

# Read all tabs into a dictionary of DataFrames
sheets_dict = pd.read_excel(excel_file, sheet_name=None)

# Iterate through each sheet (tab_name) and its data (df)
for tab_name, df in sheets_dict.items():
    
    # Clean data: Convert x, y, z to numeric
    for col in ['x', 'y', 'z']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # Drop any rows that are missing coordinates
    df_clean = df.dropna(subset=['x', 'y', 'z'])
    
    # Create a title from the tab names
    title_text = f'MNI Network: {tab_name.title()}'
    
    # Extract coordinates
    coords = df_clean[['x', 'y', 'z']].values
    
    # Plot markers onto an MNI152 glass brain
    if len(coords) > 0:
        # Nilearn requires 'node_values' to color-code the points. 
        # Assign "1"" to every point so they are all the same color
        node_vals = [1] * len(coords)
        
        display = plotting.plot_markers(
            node_values=node_vals, 
            node_coords=coords,
            title=title_text,
            node_size=80,
            node_cmap='viridis',
            display_mode='ortho', 
            alpha=0.8
        )
        plotting.show()
    else:
        print(f"No valid coordinates found to plot for {title_text}.")

## Construct Streamlines from Diffusion-Weighted Images

Requires the following BrainSuite output files from Homework 3 to be present in the current directory:

1. `"Sample.dwi.RAS.correct.nii.gz"`
2. `"Sample.dwi.RAS.mask.nii.gz"`
3. `"Sample.bval"`
4. `"Sample.dwi.RAS.bvec"`

Imports

In [ ]:
import nibabel as nib
import numpy as np
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.dti import TensorModel, fractional_anisotropy
from dipy.tracking import utils
from dipy.tracking.local_tracking import LocalTracking
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
from dipy.tracking.streamline import Streamlines
from dipy.io.stateful_tractogram import Space, StatefulTractogram
from dipy.io.streamline import save_tractogram
from dipy.direction import peaks_from_model
from dipy.data import default_sphere

Load BrainSuite output files

In [ ]:
print("Loading data...")
# BDP's corrected 4D diffusion image
img = nib.load("Sample.dwi.RAS.correct.nii.gz")
data = img.get_fdata()
affine = img.affine

# The brain mask (what is brain vs background)
mask_img = nib.load("Sample.dwi.RAS.mask.nii.gz")
mask = mask_img.get_fdata() > 0

# b-values and rotated b-vectors
bvals, bvecs = read_bvals_bvecs("Sample.bval", "Sample.dwi.RAS.bvec")
gtab = gradient_table(bvals=bvals, bvecs=bvecs)

Fit Diffusion Tensor Model

In [ ]:
print("Fitting Diffusion Tensor Model...")
# This calculates the primary direction of water movement in every single voxel
tensor_model = TensorModel(gtab)
tensor_fit = tensor_model.fit(data, mask=mask)

# Calculate Fractional Anisotropy (FA)
# FA is a measure of how "directional" the flow is (0 = sphere, 1 = perfect line)
fa = fractional_anisotropy(tensor_fit.evals)

Set up Tractography Rules

In [ ]:
print("Setting up tractography rules...")

# Rule A: Where do we stop? 
# We stop tracking if FA drops below 0.2 (usually means we left white matter and hit grey matter/CSF)
stopping_criterion = ThresholdStoppingCriterion(fa, 0.2)

# Rule B: Where do we start?
# We place a "seed" (starting point) in every voxel where the brain mask exists
seeds = utils.seeds_from_mask(mask, affine=affine, density=1)

Generate all streamlines

In [ ]:
print("Extracting tensor peaks for tracking...")
# This packages tensor data into the proper 'DirectionGetter' object
tensor_peaks = peaks_from_model(
    model=tensor_model,
    data=data,
    sphere=default_sphere, # A standard 3D sphere used to map directions
    relative_peak_threshold=0.5,
    min_separation_angle=25,
    mask=mask
)

print("Computing streamlines (this may take a minute)...")
# Pass 'tensor_peaks' instead of the raw numpy array
streamlines_generator = LocalTracking(
    direction_getter=tensor_peaks, 
    stopping_criterion=stopping_criterion,
    seeds=seeds,
    affine=affine,
    step_size=0.5 
)

# Convert the generator into an actual list of coordinates in memory
streamlines = Streamlines(streamlines_generator)
print(f"Generated {len(streamlines)} total streamlines!")

Bundle all streamlines and save to a `.trk` file

In [ ]:
print("Saving to TrackVis (.trk) format...")
# A StatefulTractogram bundles the streamlines with the anatomical image dimensions
sft = StatefulTractogram(streamlines, img, Space.RASMM)

# Save as a universal .trk file
save_tractogram(sft, "my_streamlines.trk", bbox_valid_check=False)

## Visualize Whole-Brain Tractogram

Requires `"my_streamlines.trk"` to be present in the current directory.

Imports

In [ ]:
from dipy.io.streamline import load_tractogram
from dipy.viz import window, actor, colormap
from IPython.display import Image
import os
import cv2
import numpy as np

Load in Tractogram

In [ ]:
print("Loading streamlines...")
# Load the .trk file that was created in the previous step 
# "same" tells it to use the spatial reference saved inside the file.
tractogram = load_tractogram("my_streamlines.trk", "same")
streamlines = tractogram.streamlines

Create a 3D scene and add all streamlines with a colormap.

The colormap is based on direction of the streamline.

In [ ]:
print("Preparing 3D scene...")
# Create a blank 3D canvas (Scene)
scene = window.Scene()
scene.background((0.0, 0.0, 0.0)) # completely black background

# Color the streamlines based on their direction
# Standard neuroimaging colors: Red = Left/Right, Green = Front/Back, Blue = Up/Down
colors = colormap.line_colors(streamlines)

# Create an "actor" (the 3D object that gets rendered)
streamline_actor = actor.line(lines=streamlines, colors=colors)

# Add the actor to the scene
scene.add(streamline_actor)

Open the 3D viewer

In [ ]:
print("Opening 3D viewer! You can click and drag to rotate.")
# This opens the interactive window. 
# script will pause here until 3D viewer window is closed
window.show(scene, size=(800, 800), reset_camera=True)

## Register the SUIT Cerebellar Atlas to the Subject's MRI scan

Requires the following BrainSuite output files from Homework 3 to be present in the current directory:

1. `"Sample.bfc.nii.gz"`
2. `"atl-Anatom_space-MNI_dseg.nii"`

Imports

In [ ]:
import nibabel as nib
import numpy as np
from nilearn import datasets
from dipy.align.imaffine import (transform_centers_of_mass, AffineMap,
                                 MutualInformationMetric, AffineRegistration)
from dipy.align.transforms import (RigidTransform3D, AffineTransform3D)

Load subject brain scan files

In [ ]:
print("Loading Images...")
# subject's brain (The target we want to match)
subject_img = nib.load("Sample.bfc.nii.gz")
subject_data = subject_img.get_fdata()
subject_affine = subject_img.affine

# The MNI Template (download with nilearn)
mni_img = datasets.load_mni152_template(resolution=2) 
mni_data = mni_img.get_fdata()
mni_affine = mni_img.affine

# The SUIT Cerebellar Atlas (also in MNI space)
atlas_img = nib.load('atl-Anatom_space-MNI_dseg.nii')
atlas_data = atlas_img.get_fdata()
atlas_affine = atlas_img.affine

Use mutual information to compare the two brain images

In [ ]:
print("Setting up the Registration Engine...")
metric = MutualInformationMetric(nbins=32, sampling_proportion=None)
affreg = AffineRegistration(metric=metric, level_iters=[10, 10, 5], 
                            sigmas=[3.0, 1.0, 0.0], factors=[4, 2, 1])

Align Center of Mass

In [ ]:
print("Aligning Centers of Mass...")
c_of_mass = transform_centers_of_mass(subject_data, subject_affine, 
                                      mni_data, mni_affine)

Perform Rigid (linear) Registration (Rotate & Translate)

In [ ]:
print("Performing Rigid Registration")
rigid_map = affreg.optimize(static=subject_data, moving=mni_data, transform=RigidTransform3D(), params0=None,
                            static_grid2world=subject_affine, moving_grid2world=mni_affine, starting_affine=c_of_mass.affine)

Peform Affine Registration (Scaling & Shearing)

This is a "linear warp"

In [ ]:
print("Performing Affine Registration (Scaling & Shearing)...")
affine_map = affreg.optimize(static=subject_data, moving=mni_data, transform=AffineTransform3D(), params0=None,
                             static_grid2world=subject_affine, moving_grid2world=mni_affine, starting_affine=rigid_map.affine)

Apply warp to cerebellar atlas

In [ ]:
print("Applying Warp to the Cerebellar Atlas...")
warped_atlas_data = affine_map.transform(
    atlas_data,  
    image_grid2world=atlas_affine, # prevents scaling explosion
    interpolation='nearest'
)

Save aligned (linearly warped) atlas to file

In [ ]:
print("Saving the newly aligned atlas!")
warped_atlas_img = nib.Nifti1Image(warped_atlas_data.astype(np.int32), subject_affine)
nib.save(warped_atlas_img, "Native_Cerebellar_Atlas.nii.gz")

# Intersect Regions-of-Interest (ROIs) with each Network

Requires the following files to be present in the current directory

1. `"atl-Anatom.tsv"` (downloaded from https://github.com/DiedrichsenLab/cerebellar_atlases/tree/master/Diedrichsen_2009)
2. `"loi_coordinates.xlsx"`
3. `"my_streamlines.trk"`
4. `"Native_Cerebellar_Atlas.nii.gz"`
5. `"Sample.bfc.nii.gz"`

Imports

In [ ]:
import nibabel as nib
import numpy as np
from dipy.io.streamline import load_tractogram
from dipy.tracking import utils
from dipy.viz import window, actor, colormap, ui
from dipy.align.imaffine import (transform_centers_of_mass, MutualInformationMetric, AffineRegistration)
from dipy.align.transforms import (RigidTransform3D, AffineTransform3D)
from nilearn import datasets
import pandas as pd
import csv
import os
import re
import cv2
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

Load in labels for Cerebellar Atlas

In [ ]:
def hex_to_rgb_normalized(hex_color):
    """Converts a hex color string like '#ccff00' into a tuple of RGB
    values scaled between 0.0 and 1.0"""
    hex_color = hex_color.lstrip('#')   # remove hashtag

    # parse hex pairs to integers and divide by 255.0
    r = int(hex_color[0:2], 16) / 255.0
    g = int(hex_color[2:4], 16) / 255.0
    b = int(hex_color[4:6], 16) / 255.0

    return (r, g, b)

def load_tsv_data(filepath):
    """
    Loads the TSV file and returns a list of tuples formatted as:
    (int_index, str_name, (float_r, float_g, float_b))
    """
    regions = []

    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')  # use the csv reader with a tab delimiter
        next(reader, None)  # skip the header row

        for row in reader:
            if not row:     # skip empty rows if there are any
                continue
            
            idx, name, color = row  # separate parts of row

            # convert values to their proper format
            idx = int(idx)
            color = hex_to_rgb_normalized(color)

            # append region to list of regions
            regions.append((idx, name, color))
    
    return regions

atlas_region_names_filepath = 'atl-Anatom.tsv'

User Inputs:

1. `ROIs`: integer id's of cerebellar regions of interest
2. `RADIUS_MM`: radius around each Peak Voxel Coordinate (LOI) in millimeters to turn it into a Sphere-of-Interest

In [ ]:
ROIs = [3, 4, 14, 16]
RADIUS_MM = 5  # radius of each LOI (location of interest)

Print id's and names of all cerebellar atlas regions

In [ ]:
all_regions = load_tsv_data(atlas_region_names_filepath)    # get all regions

print("All Cerebellar Regions")
print("======================")
print()
print(f"{'ID':5}{'Name'}")
print("------------------")
for id, name, color in all_regions:
    print(f"{str(id):5}{name}")

Get Peak Voxel Coordinates (Locations of Interest (LOIs)).

These are MNI coordinates of voxels with functional and structural connectivity abnormalities (found in the literature) for motor, cognitive, and emotional network.

In [ ]:
def load_locations_of_interest(filepath):
    # read all sheets from the excel file
    # setting sheet_name=None loads all sheets into a dictionary 
        # where the keys are the sheet names and values are the DataFrames.
    all_sheets = pd.read_excel(filepath, sheet_name=None)

    dataframes_to_combine = []

    # iterate through the dictionary to add the 'sheet_name' column to each sheet
    for sheet, df in all_sheets.items():
        df['sheet_name'] = sheet
        dataframes_to_combine.append(df)
    
    # concatenate all sheets into one DataFrame
    combined_df = pd.concat(dataframes_to_combine, ignore_index=True)

    # add 'unique_id' column starting at 0 and counting up
    combined_df['unique_id'] = range(len(combined_df))

    filtered_df = combined_df   # filter if desired

    # rearrange columns
    ordered_columns = [
        'sheet_name', 
        'unique_id', 
        'paper', 
        'anatomical_region', 
        'x', 
        'y', 
        'z', 
        'finding', 
        'verified'
    ]
    return filtered_df[ordered_columns] 

LOIs_filepath = 'loi_coordinates.xlsx'
LOIs_df = load_locations_of_interest(LOIs_filepath)

LOI_coords = []     # holds (x,y,z) tuples of coordinates

def is_numeric(val):
    try:
        float(val)
        return True
    except ValueError:
        return False
    
for index, row in LOIs_df.iterrows():
    coord = (row['x'], row['y'], row['z'])

    # make sure all coordinates are numeric
    all_numeric = True
    for num in coord:
        if not is_numeric(num):
            all_numeric = False
            break
    if all_numeric:
        LOI_coords.append((row['x'], row['y'], row['z']))

# display dataframe
LOIs_df

Load core data

In [ ]:
print("Loading streamlines and Native Cerebellar Atlas...")
tractogram = load_tractogram("my_streamlines.trk", "same")
streamlines = tractogram.streamlines

atlas_img = nib.load("Native_Cerebellar_Atlas.nii.gz")
atlas_data = atlas_img.get_fdata()
atlas_affine = atlas_img.affine

subject_img = nib.load("Sample.bfc.nii.gz")
subject_data = subject_img.get_fdata()
subject_affine = subject_img.affine

Calculate MNI-to-Native Coordinate System Mapping

In [ ]:
print("Calculating MNI-to-Native transformation matrix...")
mni_img = datasets.load_mni152_template(resolution=2) 
mni_data = mni_img.get_fdata()
mni_affine = mni_img.affine

metric = MutualInformationMetric(nbins=32, sampling_proportion=None)
affreg = AffineRegistration(metric=metric, level_iters=[10, 10, 5], 
                            sigmas=[3.0, 1.0, 0.0], factors=[4, 2, 1])

c_of_mass = transform_centers_of_mass(subject_data, subject_affine, mni_data, mni_affine)
rigid_map = affreg.optimize(subject_data, mni_data, RigidTransform3D(), None,
                            static_grid2world=subject_affine, moving_grid2world=mni_affine, starting_affine=c_of_mass.affine)
affine_map = affreg.optimize(subject_data, mni_data, AffineTransform3D(), None,
                             static_grid2world=subject_affine, moving_grid2world=mni_affine, starting_affine=rigid_map.affine)

Class for storing streamline counts

In [ ]:
class StreamlineCounts:
    def __init__(self):
        """
        self.whole_brain (int): number of streamlines in the entire brain
        self.ROI_LOI_dict (dict[str, dict[int, int]]): keys = ROI name (str) ; values (dict) = keys are integers corresponding to the 'unique_id' of a LOI, 
            which give the number of streamlines that intersect both the ROI and LOI 
        self.regions_dict (dict[str, int]): keys = ROI name (str) ; values = total number of streamlines intersecting the ROI
        """
        self.whole_brain = 0
        self.ROI_LOI_dict = dict()
        self.regions_dict = dict()

Function to Intersect each network with each ROI and return a `StreamlineCounts` object with the calculations.

In [ ]:
def intersect_ROIs_and_networks():
    # Get a list of all unique sheet names in your dataframe
    unique_sheets = LOIs_df['sheet_name'].unique()

    # create an object to store all streamline counts
    streamline_counts = StreamlineCounts()
    streamline_counts.whole_brain = len(streamlines)

    for target_region_id in ROIs:   # for each Cerebellar Region
        roi_name = next((name for idx, name, color in all_regions if idx == target_region_id), f"Region_{target_region_id}")
        
        # Initialize tracking for this ROI
        streamline_counts.regions_dict[roi_name] = 0
        streamline_counts.ROI_LOI_dict[roi_name] = dict()

        # Calculate ROI tracts once (Variable 'b')
        region_mask = atlas_data == target_region_id
        all_region_tracts = list(utils.target(streamlines, atlas_affine, region_mask, include=True))
        b_count = len(all_region_tracts)
        streamline_counts.regions_dict[roi_name] = b_count
        
        for current_sheet in unique_sheets:
            print(f"ROI: {roi_name} | Network: {current_sheet}")
            
            # Process Network LOIs & Create Network Stats
            sheet_df = LOIs_df[LOIs_df['sheet_name'] == current_sheet]
            shape = subject_img.shape
            
            # store the union of all spheres in the network to calculate 'a'
            network_mask = np.zeros(shape, dtype=bool)

            for index, row in sheet_df.iterrows():
                coord = (row['x'], row['y'], row['z'])
                
                # make sure all coordinates are numeric
                all_numeric = True
                for num in coord:
                    if not is_numeric(num):
                        all_numeric = False
                        break
                if not all_numeric:
                    continue    # move to next LOI

                MNI_X, MNI_Y, MNI_Z = coord
                unique_id = int(row['unique_id'])

                # Map to Native
                mni_point = np.array([MNI_X, MNI_Y, MNI_Z, 1.0])
                native_point = np.linalg.inv(affine_map.affine).dot(mni_point)[:3]
                
                # Create sphere mask for this specific LOI
                sphere_mask = np.zeros(shape, dtype=bool)
                center_vox = nib.affines.apply_affine(np.linalg.inv(subject_affine), native_point)
                vox_sizes = nib.affines.voxel_sizes(subject_affine)
                rx, ry, rz = [int(np.ceil(RADIUS_MM / s)) for s in vox_sizes]
                cx, cy, cz = int(center_vox[0]), int(center_vox[1]), int(center_vox[2])

                for i in range(max(0, cx-rx), min(shape[0], cx+rx+1)):
                    for j in range(max(0, cy-ry), min(shape[1], cy+ry+1)):
                        for k in range(max(0, cz-rz), min(shape[2], cz+rz+1)):
                            p = nib.affines.apply_affine(subject_affine, [i, j, k])
                            if np.linalg.norm(p - native_point) <= RADIUS_MM:
                                sphere_mask[i, j, k] = True
                                # OR this into the total network mask
                                network_mask[i, j, k] = True

                # Calculate specific LOI connection for the count dict
                loi_tracts = list(utils.target(all_region_tracts, subject_affine, sphere_mask, include=True))
                streamline_counts.ROI_LOI_dict[roi_name][unique_id] = len(loi_tracts)

            # Calculate 'a' (Unique streamlines hitting ROI AND any point in Network)
            network_tracts = list(utils.target(all_region_tracts, subject_affine, network_mask, include=True))
            a_count = len(network_tracts)

    return streamline_counts

# call function
streamline_counts = intersect_ROIs_and_networks()

### Produce Animations

**Warning: It takes about ~2-4 minutes to produce every 100 frames.**

For each region, for each network:

* Create a dynamic animation video to visualize the cerebellar region, network Spheres-of-Interest

In [ ]:
def create_network_flyaround_animations():
    # Get a list of all unique sheet names in the dataframe
    unique_sheets = LOIs_df['sheet_name'].unique()

    for target_region_id in ROIs:   # for each Cerebellar Region
        roi_name = next((name for idx, name, color in all_regions if idx == target_region_id), f"Region_{target_region_id}")

        # Calculate ROI tracts once (Variable 'b')
        region_mask = atlas_data == target_region_id
        all_region_tracts = list(utils.target(streamlines, atlas_affine, region_mask, include=True))
        b_count = len(all_region_tracts)
        streamline_counts.regions_dict[roi_name] = b_count
        
        for current_sheet in unique_sheets:
            print(f"\n{'='*60}")
            print(f"Building Video | ROI: {roi_name} | Network: {current_sheet}")
            
            scene = window.Scene()
            scene.background((0.0, 0.0, 0.0))   # all black background

            # Background Rendering
            all_streamlines = list(utils.target(streamlines, atlas_affine, region_mask, include=False))
            if len(all_streamlines) > 0:
                all_streamlines_actor = actor.line(lines=all_streamlines, colors=(0.2, 0.5, 1.0))
                all_streamlines_actor.GetProperty().SetOpacity(0.02)
                scene.add(all_streamlines_actor)

            if b_count > 0:
                debug_actor = actor.line(lines=all_region_tracts, colors=(1.0, 1.0, 1.0))
                debug_actor.GetProperty().SetOpacity(0.05) 
                scene.add(debug_actor)

            try:
                region_actor = actor.contour_from_roi(region_mask, affine=atlas_affine, color=(25/255, 219/255, 22/255), opacity=0.6)
                scene.add(region_actor)
            except Exception as e:
                print(f"Could not draw contour for region {target_region_id}: {e}")

            # 2. Process Network LOIs & Create Network Stats
            sheet_df = LOIs_df[LOIs_df['sheet_name'] == current_sheet]
            shape = subject_img.shape
            
            # This will store the union of all spheres in the network to calculate 'a'
            network_mask = np.zeros(shape, dtype=bool)

            for index, row in sheet_df.iterrows():
                coord = (row['x'], row['y'], row['z'])
                
                # make sure all coordinates are numeric
                all_numeric = True
                for num in coord:
                    if not is_numeric(num):
                        all_numeric = False
                        break
                if not all_numeric:
                    continue    # move to next LOI

                MNI_X, MNI_Y, MNI_Z = coord
                unique_id = int(row['unique_id'])

                # Map MNI to Native space
                mni_point = np.array([MNI_X, MNI_Y, MNI_Z, 1.0])
                native_point = np.linalg.inv(affine_map.affine).dot(mni_point)[:3]
                
                # Create sphere mask for this specific LOI
                sphere_mask = np.zeros(shape, dtype=bool)
                center_vox = nib.affines.apply_affine(np.linalg.inv(subject_affine), native_point)
                vox_sizes = nib.affines.voxel_sizes(subject_affine)
                rx, ry, rz = [int(np.ceil(RADIUS_MM / s)) for s in vox_sizes]
                cx, cy, cz = int(center_vox[0]), int(center_vox[1]), int(center_vox[2])

                for i in range(max(0, cx-rx), min(shape[0], cx+rx+1)):
                    for j in range(max(0, cy-ry), min(shape[1], cy+ry+1)):
                        for k in range(max(0, cz-rz), min(shape[2], cz+rz+1)):
                            p = nib.affines.apply_affine(subject_affine, [i, j, k])
                            if np.linalg.norm(p - native_point) <= RADIUS_MM:
                                sphere_mask[i, j, k] = True
                                # OR this into the total network mask
                                network_mask[i, j, k] = True

                # Calculate specific LOI connection for the count dict
                loi_tracts = list(utils.target(all_region_tracts, subject_affine, sphere_mask, include=True))

                if len(loi_tracts) > 0:
                    scene.add(actor.line(lines=loi_tracts, colors=(255/255, 165/255, 0/255)))
                scene.add(actor.sphere(centers=np.array([native_point]), colors=(1, 0, 0), radii=RADIUS_MM))

            # Calculate 'a' (Unique streamlines hitting ROI AND any point in Network)
            network_tracts = list(utils.target(all_region_tracts, subject_affine, network_mask, include=True))
            a_count = len(network_tracts)
            percentage = (a_count / b_count * 100) if b_count > 0 else 0
            
            # Create the stats string for the top-left corner
            stats_text = f"{a_count}/{b_count} ({percentage:.1f}%) intersect w/ ROI & network"

            # Camera Animation
            safe_sheet_name = re.sub(r'[^A-Za-z0-9_-]', '_', current_sheet)
            frame_folder = f"video_frames/ROI_{target_region_id}_{safe_sheet_name}"
            os.makedirs(frame_folder, exist_ok=True)

            scene.set_camera(position=(0, -375, 75), focal_point=(0, 0, 0), view_up=(0, 0, 1))
            frame_idx = 0
            frames_per_orbit = 120
            
            # Orbit 1 & 2 Rendering
            for _ in range(frames_per_orbit):
                scene.azimuth(360 / frames_per_orbit)
                window.record(scene=scene, out_path=os.path.join(frame_folder, f"frame_{frame_idx}.png"), size=(1200, 1200), reset_camera=False)
                frame_idx += 1

            current_pos, focal_point, _ = scene.get_camera()
            radius = np.sqrt(current_pos[1]**2 + current_pos[2]**2)
            start_angle = np.arctan2(current_pos[2], current_pos[1])

            for i in range(frames_per_orbit):
                angle = start_angle + (i * (2 * np.pi / frames_per_orbit))
                my_pos = (current_pos[0], radius * np.cos(angle), radius * np.sin(angle))
                my_up = (0, radius * np.sin(angle), -radius * np.cos(angle))
                window.record(scene=scene, cam_pos=my_pos, cam_focal=focal_point, cam_view=my_up,
                                out_path=os.path.join(frame_folder, f"frame_{frame_idx}.png"), size=(1200, 1200), reset_camera=False)
                frame_idx += 1

            # OpenCV Video Stitching (with dual text overlay)
            video_name = f"animations/Network_{safe_sheet_name}_ROI_{target_region_id}.mp4"
            images = sorted([img for img in os.listdir(frame_folder) if img.endswith(".png")], key=lambda f: int(re.sub('\D', '', f)))
            
            sample_frame = cv2.imread(os.path.join(frame_folder, images[0]))
            h, w, _ = sample_frame.shape
            video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), 20, (w, h))

            bottom_label = f"ROI: {roi_name} | Network: {current_sheet}"

            for img_name in images:
                frame = cv2.imread(os.path.join(frame_folder, img_name))
                
                # Top-Left: Network Stats (a/b %)
                cv2.putText(frame, stats_text, (40, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
                
                # Bottom-Left: Names
                cv2.putText(frame, bottom_label, (40, h - 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)

                video.write(frame)

            video.release()
            print(f"Video saved: {video_name}")
                
            scene.clear()

create_network_flyaround_animations()

Statistical Analysis of Streamline Counts

In [ ]:
def run_network_chi_square(streamline_counts, LOIs_df, region_v_name, region_viib_name, cognitive_sheets):
    """
    Processes the streamline_counts object and runs a Chi-Square Test of Independence
    to compare Motor vs Cognitive network connectivity between two cerebellar regions.
    """
    print("STATISTICAL ANALYSIS: CHI-SQUARE TEST OF INDEPENDENCE")
    print("="*50)

    # Map "unique_id" to Network Category (Motor vs Cognitive)
    motor_sheets = ["motor_control"]

    motor_ids = LOIs_df[LOIs_df['sheet_name'].isin(motor_sheets)]['unique_id'].astype(int).tolist()
    cognitive_ids = LOIs_df[LOIs_df['sheet_name'].isin(cognitive_sheets)]['unique_id'].astype(int).tolist()

    # aggregate the counts from the ROI_LOI_dict
    # Initialize the 4 buckets
    v_motor = 0
    v_cognitive = 0
    viib_motor = 0
    viib_cognitive = 0

    # process Region V
    if region_v_name in streamline_counts.ROI_LOI_dict:
        for unique_id, count in streamline_counts.ROI_LOI_dict[region_v_name].items():
            if unique_id in motor_ids:
                v_motor += count
            elif unique_id in cognitive_ids:
                v_cognitive += count

    # process Region VIIb
    if region_viib_name in streamline_counts.ROI_LOI_dict:
        for unique_id, count in streamline_counts.ROI_LOI_dict[region_viib_name].items():
            if unique_id in motor_ids:
                viib_motor += count
            elif unique_id in cognitive_ids:
                viib_cognitive += count

    # Build a contingency table
    # Format: [[row1_col1, row1_col2], [row2_col1, row2_col2]]
    contingency_table = np.array([
        [v_motor, v_cognitive],       # Row 1: Lobule V
        [viib_motor, viib_cognitive]  # Row 2: Lobule VIIb
    ])

    # Convert to pandas DataFrame just for a formatted printout
    df_table = pd.DataFrame(
        contingency_table, 
        columns=['Motor Network', f'{cognitive_sheets}'], 
        index=[region_v_name, region_viib_name]
    )
    
    print("\nRaw Streamline Connections (Contingency Table)")
    print(df_table)

    # prevent crash if a region or network has zero connections
    row_sums = contingency_table.sum(axis=1)
    col_sums = contingency_table.sum(axis=0)

    if np.any(row_sums == 0):
        empty_region = region_v_name if row_sums[0] == 0 else region_viib_name
        print(f"\n{empty_region} has ZERO connections to both of these networks. Cannot run Chi-Square.")
        return
        
    if np.any(col_sums == 0):
        empty_network = 'Motor' if col_sums[0] == 0 else 'Cognitive'
        print(f"\nThe {empty_network} target has ZERO connections from either lobule. Cannot run Chi-Square.")
        return

    # Execute the Chi-Square Test
    # chi2: The test statistic
    # p: The p-value
    # dof: Degrees of freedom
    # expected: The internally normalized frequencies the test expected to see
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # Calculate Effect Size (Cramer's V)
    # Tells us if the difference is actually meaningful (0 to 1 scale).
    n_total = contingency_table.sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n_total * min_dim))

    # Print the Results
    print("\nStatistical Results")
    print("=" * 50)
    print(f"Chi-Square Statistic: {chi2:.2f}")
    
    if p < 0.001:
        print("P-Value: < 0.001 *** (Statistically Significant)")
    else:
        print(f"P-Value: {p:.4f}")

    print(f"Cramer's V (Effect Size): {cramers_v:.4f}")
    

# run statistical analysis
all_cognitive_sheets = ["PFC", "mdTHA", "parietal_precuneus"]
sides = ["Left", "Right"]

for side in sides:
    # all cognitive sheets
    run_network_chi_square(streamline_counts=streamline_counts, LOIs_df=LOIs_df, region_v_name=f"{side}_V", region_viib_name=f"{side}_VIIb", cognitive_sheets=all_cognitive_sheets)

    for cog_sheet in all_cognitive_sheets:
            run_network_chi_square(streamline_counts=streamline_counts, LOIs_df=LOIs_df, region_v_name=f"{side}_V", region_viib_name=f"{side}_VIIb", cognitive_sheets=[cog_sheet])